# GPT-2 Playground

A from-scratch walkthrough of the building blocks of a GPT-2 style transformer.
Each section defines a module and then runs a quick shape-check on random input.

## Imports

In [33]:
from dataclasses import dataclass

import torch.nn as nn
import torch
import torch.nn.functional as F
import math

## Model Config

Hyperparameters for the model (block size, vocab size, layers, heads, embedding dim).

In [34]:
@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50257
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768

## Causal Self-Attention

Multi-head masked self-attention. Projects the input to query/key/value, splits across
heads, applies the causal mask + softmax, then recombines and projects the output.

In [35]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0 # embedding dimension must be divisible by the number of heads
        # key, query, value projection/weights for all heads, but in a batch
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        # output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        # regularization
        self.n_head = config.n_head
        print(f"n_head is: {self.n_head}")
        self.n_embd = config.n_embd
        print(f"n_embd is: {self.n_embd}")
        # Not the bias, actually is mask, following the openAI/HF naming convention
        self.register_buffer("bias", torch.tril(torch.ones(config.block_size, config.block_size))
                                 .view(1, 1, config.block_size, config.block_size))

    def forward(self, x):
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)
        print(f"B,T,C are: {B}, {T}, {C}")
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh is the number of heads, hs is "head size", and C is the embedding dimensionality = (nh * hs)
        qkv = self.c_attn(x)
        print(f"combine one qkv shape: {qkv.shape}")
        q, k, v = qkv.split(self.n_embd, dim=2)
        print(f"split one qkv shape: {q.shape}, {k.shape}, {v.shape}")
        k = k.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        print(f"k shape: {k.shape}")
        q = q.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        print(f"q shape: {q.shape}")
        v = v.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        print(f"v shape: {v.shape}")
        # attention (T,T) matrix for all the queries and keys
        att = q @ k.transpose(-2, -1) * (1.0 / math.sqrt(k.size(-1)))
        print(f"att shape: {att.shape}")
        att = att.masked_fill(self.bias[:,:,:T,:T] == 0, float('-inf'))
        # Apply softmax on last dim of attention matrix
        att = F.softmax(att,dim=-1)
        y = att @ v # (B, nh, T, hs) x (B, nh, hs, T) -> (B, nh, T, T)
        print(f"y shape: {y.shape}")
        y = y.transpose(1,2).contiguous().view(B,T,C) # re-assemble all head outputs side by side
        print(f"resemble shape: {y.shape}")
        # output projection
        y = self.c_proj(y)
        return y

### Shape Check: Self-Attention

Run a random `(1, 256, 384)` tensor through the attention block and confirm the
output shape matches the input.

In [36]:
x = torch.randn(1, 1024, 768)
print("in :", tuple(x.shape))

model = CausalSelfAttention(GPTConfig()).eval()

with torch.no_grad():
    y = model(x)

print("out:", tuple(y.shape))

in : (1, 1024, 768)
n_head is: 12
n_embd is: 768
B,T,C are: 1, 1024, 768
combine one qkv shape: torch.Size([1, 1024, 2304])
split one qkv shape: torch.Size([1, 1024, 768]), torch.Size([1, 1024, 768]), torch.Size([1, 1024, 768])
k shape: torch.Size([1, 12, 1024, 64])
q shape: torch.Size([1, 12, 1024, 64])
v shape: torch.Size([1, 12, 1024, 64])
att shape: torch.Size([1, 12, 1024, 1024])
y shape: torch.Size([1, 12, 1024, 64])
resemble shape: torch.Size([1, 1024, 768])
out: (1, 1024, 768)


## MLP (Feed-Forward)

The position-wise feed-forward network: expand to 4x the embedding dim, apply GELU,
then project back down.

In [37]:
class MLP(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

### Shape Check: MLP

Confirm the MLP preserves the input shape `(1, 256, 384)`.

In [41]:
x = torch.randn(1, 1024, 768)
print("in :", tuple(x.shape))             # (1, 256, 384)

model = MLP(GPTConfig()).eval()

with torch.no_grad():
    y = model(x)

print("out:", tuple(y.shape))             # (1, 256, 384)

in : (1, 1024, 768)
out: (1, 1024, 768)
